# 🎓 Reinforcement Learning Week 2 (Lecture 3): Stable Baselines3 Exercises (2.1 – 2.5)

**Course:** Data & A.I. 6 (DAI6)  
**Instructor:** Alexander Michielsen  
**Scope:** Complete, runnable solutions and simple-English explanations for **Exercises 2.1, 2.2, 2.3, 2.4, and 2.5** from `RL_Exercises_Week2.pdf`.

---

## 🧭 Why Use Stable Baselines3 (SB3)?

In `reinforcement_q_learning.ipynb`, we wrote nearly 200 lines of PyTorch code just to build a basic DQN agent (`ReplayMemory`, `policy_net`, `target_net`, `SmoothL1Loss`, soft updates).  
**Stable Baselines3 (SB3)** gives you tested, production-ready Deep RL algorithms (`DQN`, `PPO`, `A2C`, `SAC`, `TD3`, `DDPG`) in **3 lines of code**, making it easy to swap algorithms and compare them fairly.

```bash
# Install required packages if not yet installed:
# pip install "stable-baselines3[extra]" gymnasium[classic_control] flappy-bird-gymnasium tensorboard
```

In [ ]:
import gymnasium as gym
import numpy as np
import torch
from stable_baselines3 import A2C, DQN, PPO, SAC
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback

print("Gymnasium version:", gym.__version__)
print("PyTorch version:", torch.__version__)

---

# 🕹️ Exercise 2.1: Playing Around in Stable Baselines3

### 1. Train `CartPole-v1` using `A2C` vs. `DQN` (100,000 steps)

| Metric Printed in Terminal | Simple Meaning |
| :--- | :--- |
| **`rollout/ep_len_mean`** | Average number of steps the pole stayed upright over the last 100 episodes (max is `500`). |
| **`rollout/ep_rew_mean`** | Average total score over the last 100 episodes (`+1` per step $\rightarrow$ max is `500`). |
| **`time/fps`** | How many simulation steps per second your computer is running. |
| **`train/loss` or `value_loss`** | How big the neural network's prediction errors currently are. |

### Which Algorithm Performs Better (`A2C` vs `DQN`)?
* With default SB3 settings over 100,000 steps, **`A2C` (and `PPO`) usually reaches the maximum score of `500` much more smoothly and reliably** than default `DQN`.
* **Why?** SB3's default `DQN` hyperparameters are tuned for complex Atari games (slow exploration decay, small learning rate), so on tiny `CartPole-v1` it can oscillate unless tuned.

In [ ]:
# 1A. Train A2C on CartPole-v1
env_a2c = gym.make("CartPole-v1")
model_a2c = A2C("MlpPolicy", env_a2c, verbose=1, seed=42)
model_a2c.learn(total_timesteps=25_000)  # Set to 100_000 for full run

mean_a2c, std_a2c = evaluate_policy(model_a2c, env_a2c, n_eval_episodes=10, deterministic=True)
print(f"A2C Evaluation Score: {mean_a2c:.1f} +/- {std_a2c:.1f}")
env_a2c.close()

In [ ]:
# 1B. Train DQN on CartPole-v1 (simply replacing A2C with DQN)
env_dqn = gym.make("CartPole-v1")
model_dqn = DQN("MlpPolicy", env_dqn, verbose=1, seed=42)
model_dqn.learn(total_timesteps=25_000)  # Set to 100_000 for full run

mean_dqn, std_dqn = evaluate_policy(model_dqn, env_dqn, n_eval_episodes=10, deterministic=True)
print(f"DQN Evaluation Score: {mean_dqn:.1f} +/- {std_dqn:.1f}")
env_dqn.close()

### 2. Understanding Action/Observation `Spaces` — Will `DDPG` or `SAC` Work on `CartPole-v1`?

| Gymnasium Space | What It Represents | Example |
| :--- | :--- | :--- |
| **`Discrete(n)`** | A fixed set of $n$ whole-number buttons: `{0, 1, ..., n-1}`. | `CartPole-v1` Action Space: `Discrete(2)` (`0` = Left, `1` = Right). |
| **`Box(low, high, shape)`** | Continuous decimal numbers inside `[low, high]`. | `CartPole-v1` Observation Space: `Box(4,)` (4 decimal measurements). |
| **`MultiDiscrete`** | Multiple discrete dials/buttons chosen at once. | Picking movement (`4` choices) + jump (`2` choices) together. |
| **`MultiBinary`** | Multiple on/off (`0`/`1`) switches. | Pressing combinations of keyboard keys. |

#### Question: Will `DDPG` or `SAC` work on `CartPole-v1`?
* **Answer: ❌ NO!**
* **Why?** `CartPole-v1` has a **`Discrete(2)`** action space, whereas **`DDPG` and `SAC` only work on continuous `Box` action spaces!** Let's verify this in code below:

In [ ]:
# Verify that SAC raises an error on Discrete(2) CartPole-v1
env_test = gym.make("CartPole-v1")
print("CartPole Observation Space:", env_test.observation_space)
print("CartPole Action Space:", env_test.action_space)

try:
    SAC("MlpPolicy", env_test)
except AssertionError as e:
    print("\nExpected Error when trying SAC on CartPole-v1:")
    print(e)
env_test.close()

### 3. Using Callbacks (Checkpointing & Saving the Best Model)
* **`CheckpointCallback`:** Saves a backup `.zip` file every $N$ steps so you never lose a long training run.
* **`EvalCallback`:** Tests the agent periodically and saves `best_model.zip` whenever it sets a new high score.

In [ ]:
checkpoint_callback = CheckpointCallback(
    save_freq=5_000,
    save_path="./checkpoints/",
    name_prefix="cartpole_a2c"
)
print("CheckpointCallback configured to save every 5,000 steps.")

---

# 📈 Exercise 2.2: TensorBoard Logging & Custom Reward Wrapper

### 1. Custom Callback to Log the Pole Angle to TensorBoard
In `CartPole-v1`, the 4 state numbers are:
`[0: cart_pos, 1: cart_vel, 2: pole_angle, 3: pole_angular_vel]`.

Below, we write a custom `BaseCallback` that reads `obs[2]` (the pole angle in radians) at every step and logs it to TensorBoard under `custom/pole_angle`:

In [ ]:
class PoleAngleTensorboardCallback(BaseCallback):
    """Custom callback that logs the CartPole pole angle to TensorBoard."""
    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self) -> bool:
        # self.locals['new_obs'] has shape (n_envs, 4)
        obs = self.locals["new_obs"][0]
        pole_angle_rad = float(obs[2])
        pole_angle_deg = np.degrees(pole_angle_rad)
        
        self.logger.record("custom/pole_angle_rad", pole_angle_rad)
        self.logger.record("custom/pole_angle_deg", pole_angle_deg)
        return True

# Example usage with TensorBoard logging:
env_tb = gym.make("CartPole-v1")
model_tb = PPO("MlpPolicy", env_tb, tensorboard_log="./tb_logs/", verbose=0, seed=42)
model_tb.learn(total_timesteps=10_000, callback=PoleAngleTensorboardCallback(), tb_log_name="PPO_PoleAngle")
env_tb.close()
print("Logged training & custom pole angle to ./tb_logs/ (View with: tensorboard --logdir ./tb_logs/)")

### 2. Custom `RewardWrapper`: `0` while upright, `-1` when it falls

#### Question: How does changing the reward from `+1 per step` to `0 when upright, -1 when falling` affect training?

| Reward Design | Return $G_0$ if Pole Falls at Step $T$ | Return $G_0$ if Pole Survives All 500 Steps | Impact on Training |
| :--- | :--- | :--- | :--- |
| **Default (`+1` per step)** | $\sum_{t=0}^{T-1} \gamma^t (1)$ (Higher for larger $T$) | Maximum positive return | **Dense feedback:** Every single step immediately rewards survival. |
| **Custom (`0` upright, `-1` on fall)** | $\gamma^{T-1} (-1)$ (e.g. falling at step 5 gives $-0.96$; falling at step 200 gives $-0.13$) | **`0.0`** (Since episode is `truncated`, not `terminated`!) | **Still learns, but slower at first:** Because $\gamma = 0.99 < 1$, the agent still wants to delay the `-1` crash as long as possible (or avoid it completely by reaching step 500 for `0.0`). However, receiving `0` on every upright step makes early credit assignment sparser. |

In [ ]:
class ZeroUprightMinusOneFallWrapper(gym.Wrapper):
    """Gives 0.0 reward while upright and -1.0 when the pole falls (terminated)."""
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        custom_reward = -1.0 if terminated else 0.0
        return obs, custom_reward, terminated, truncated, info

# Train PPO on the custom-reward CartPole, then evaluate on standard CartPole to check true survival duration
train_env = ZeroUprightMinusOneFallWrapper(gym.make("CartPole-v1"))
eval_env = gym.make("CartPole-v1")

model_custom_rew = PPO("MlpPolicy", train_env, verbose=0, seed=42)
model_custom_rew.learn(total_timesteps=30_000)

mean_steps, std_steps = evaluate_policy(model_custom_rew, eval_env, n_eval_episodes=10, deterministic=True)
print(f"Agent trained with (0 upright / -1 fall) survived on average: {mean_steps:.1f} +/- {std_steps:.1f} steps!")
train_env.close()
eval_env.close()

---

# ⛰️ Exercise 2.3: Continuous Environments (`MountainCarContinuous-v0`)

### Questions & Answers:
1. **Can you use `DQN` for `MountainCarContinuous-v0`? Why (not)?**
   * **❌ No!** In `MountainCarContinuous-v0`, the action is a continuous real number between `-1.0` and `+1.0` (`Box(-1.0, 1.0, (1,))`) representing how hard the car pushes left or right.
   * `DQN` requires one output neuron per discrete action to compute $\max_a Q(s, a)$. You cannot have infinitely many output neurons!
2. **Which algorithm would you use instead?**
   * Any continuous-control Actor-Critic algorithm: **`SAC` (Soft Actor-Critic)**, **`TD3`**, **`DDPG`**, or **`PPO`**.

In [ ]:
env_mc = gym.make("MountainCarContinuous-v0")
print("MountainCarContinuous Observation Space:", env_mc.observation_space)
print("MountainCarContinuous Action Space:", env_mc.action_space)

# Train using SAC (designed for continuous Box action spaces)
model_sac = SAC("MlpPolicy", env_mc, verbose=0, seed=42)
model_sac.learn(total_timesteps=5_000)  # Increase to 50_000+ to fully solve
print("SAC successfully trained on continuous action space!")
env_mc.close()

---

# ⚖️ Exercise 2.4: Comparing `DQN` and `PPO` Fairly

### Complete Answers to Parts `a` – `f`:

| Sub-question | Answer |
| :--- | :--- |
| **a. Maximum possible reward in `CartPole-v1`?** | **`500.0`** (`+1` per step, and the environment truncates at `max_episode_steps = 500`). |
| **b. How many input and output neurons in `model.q_net`?** | **`4` input neurons** (for the 4 state values) and **`2` output neurons** (one Q-value for Left, one for Right). |
| **c. Which performs better after 50,000 steps (`DQN` vs `PPO`)?** | **`PPO`** reliably reaches `500.0 ± 0.0`, whereas default SB3 `DQN` has much higher variance at 50,000 steps. |
| **d. Is a single run enough to conclude one algorithm is better?** | **❌ No!** RL training has high randomness (weight initialization, environment resets, exploration). You must always test across **multiple random seeds (`seed=0, 1, 2`)** and report `mean ± std`. |
| **e. What does `deterministic=True` do during evaluation, and why use it?** | It disables random exploration (turns off $\varepsilon$-greedy in DQN and action sampling in PPO) so the agent strictly picks its highest-value action, measuring its **true learned policy** rather than random luck. |

In [ ]:
# Part b: Inspect model.q_net input and output neurons
env_inspect = gym.make("CartPole-v1")
dqn_inspect = DQN("MlpPolicy", env_inspect, seed=0)
print("=== Part (b): DQN Neural Network Architecture (model.q_net) ===")
print(dqn_inspect.q_net)
env_inspect.close()

In [ ]:
# Parts a, c, d, e: Compare DQN and PPO across Seeds 0, 1, 2 (50,000 steps each)
TIMESTEPS = 50_000
SEEDS = [0, 1, 2]
results_table = {"DQN": {}, "PPO": {}}

for algo_name, AlgoClass in [("DQN", DQN), ("PPO", PPO)]:
    for s in SEEDS:
        train_env = gym.make("CartPole-v1")
        eval_env = gym.make("CartPole-v1")
        
        model = AlgoClass("MlpPolicy", train_env, seed=s, verbose=0)
        model.learn(total_timesteps=TIMESTEPS)
        
        mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
        results_table[algo_name][s] = f"{mean_r:.1f} ± {std_r:.1f}"
        print(f"{algo_name} | Seed {s} -> Mean Reward: {mean_r:.1f} ± {std_r:.1f}")
        
        train_env.close()
        eval_env.close()

print("\n=== Final Comparison Table (Part d) ===")
print(f"{'Algorithm':<10} | {'Seed 0':<18} | {'Seed 1':<18} | {'Seed 2':<18}")
print("-" * 72)
for algo_name in ["DQN", "PPO"]:
    print(f"{algo_name:<10} | {results_table[algo_name][0]:<18} | {results_table[algo_name][1]:<18} | {results_table[algo_name][2]:<18}")

--- 

# 🐦 Exercise 2.5: Flappy Bird (`flappy-bird-gymnasium`)

In `FlappyBird-v0`:
* **Actions (`Discrete(2)`):** `0` = Do nothing (glide down), `1` = Flap wings!
* **Observations:**
  * `use_lidar=False` (Recommended): Gives **12 clean features** (distances to the next pipes, bird vertical position, bird vertical velocity).
  * `use_lidar=True`: Gives 180 LIDAR distance rays.

In [ ]:
try:
    import flappy_bird_gymnasium
    env_fb = gym.make("FlappyBird-v0", use_lidar=False)
    print("FlappyBird Observation Space (12 features):", env_fb.observation_space)
    print("FlappyBird Action Space (0=Idle, 1=Flap):", env_fb.action_space)
    
    model_fb = PPO("MlpPolicy", env_fb, verbose=1, seed=42)
    model_fb.learn(total_timesteps=20_000)  # Train for 300k-500k steps for high scores
    env_fb.close()
except ImportError:
    print("Run `pip install flappy-bird-gymnasium` to train on FlappyBird-v0!")